# Phân Tích EDA Biến Phân Loại
## Tập Dữ Liệu Dự Báo Hủy Đặt Phòng Khách Sạn

**Dự án**: ML Pipeline Dự Báo Hủy Đặt Phòng Khách Sạn  
**Thành viên**: Thành viên 3 - Phân tích dữ liệu phân loại  
**Giai đoạn**: Giai đoạn 1 - Nạp dữ liệu & EDA  
**Mục tiêu**: Phân tích các biến phân loại và mối quan hệ của chúng với biến mục tiêu `is_canceled`

---

### Tổng Quan
Notebook này thực hiện phân tích khám phá dữ liệu (EDA) toàn diện trên các biến phân loại từ tập dữ liệu đặt phòng khách sạn. Chúng ta sẽ:
- Phân tích sự phân phối và đa dạng của các biến phân loại
- Kiểm tra mối quan hệ giữa các biến phân loại và kết quả hủy đặt phòng
- Tính toán độ mạnh liên hệ bằng kiểm định thống kê (Chi-Square, Cramér's V)
- Xác định biến phân loại nào có khả năng dự đoán hủy phòng tốt nhất


## Phần 1: Tải Dữ Liệu & Tổng Quan Nhanh
Tải tập dữ liệu đặt phòng khách sạn và có cái nhìn tổng quan ban đầu về tất cả các biến, đặc biệt tập trung vào các biến phân loại.


In [3]:
# ==== PHẦN 0: Import & Cài Đặt ====

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Import các module từ dự án
from config import DATA_CONFIG,EDA_CONFIG
from modules.data_loader import load_data_pipeline
from modules.eda import (
    analyze_categorical_features_distribution,
    analyze_categorical_target_relationship,
    plot_categorical_distribution,
    plot_categorical_with_target,
    plot_categorical_association_heatmap,
    run_categorical_eda,
    cramers_v
)

# Cấu hình matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Import thành công!")
print("✓ Các module dự án đã được nạp thành công!")

✓ Import thành công!
✓ Các module dự án đã được nạp thành công!


## Phần 2: Chạy Pipeline EDA Biến Phân Loại Hoàn Chỉnh

Chúng ta sẽ chạy toàn bộ pipeline phân tích, bao gồm:
1. Phân tích phân phối (số giá trị khác nhau, số lượng từng giá trị, tỷ lệ thiếu)
2. Phân tích mối quan hệ với biến mục tiêu (tỷ lệ hủy, kiểm định chi-square, Cramér's V)
3. Trực quan hóa (biểu đồ phân phối từng đặc trưng, tỷ lệ hủy theo danh mục, heatmap mức độ liên kết)


In [4]:
# Tải dữ liệu bằng pandas (file đã có sẵn ở local)
# Xây dựng đường dẫn tương đối so với vị trí notebook
import os
notebook_dir = os.path.dirname(os.path.abspath("categorical_eda_analysis.ipynb"))
data_path = os.path.join(notebook_dir, '..', DATA_CONFIG['file_path'])
data_path = os.path.normpath(data_path)

try:
    df = pd.read_csv(data_path)
    print(f"✓ Tải dữ liệu thành công!")
    print(f"  Kích thước: {df.shape} (dòng, cột)")
    print(f"  Đường dẫn: {data_path}")
except FileNotFoundError:
    # Thử đường dẫn dự phòng
    try:
        df = pd.read_csv('../data/raw_data.csv')
        print(f"✓ Tải dữ liệu thành công từ ../data/raw_data.csv")
        print(f"  Kích thước: {df.shape} (dòng, cột)")
    except FileNotFoundError as e:
        print(f"Lỗi: Không tìm thấy file dữ liệu. Đã thử: {data_path}")
        print(f"Thư mục làm việc hiện tại: {os.getcwd()}")
        raise

# Hiển thị thông tin cơ bản
print("\n" + "="*80)
print("TỔNG QUAN VỀ TẬP DỮ LIỆU")
print("="*80)
print(f"Tổng số mẫu: {len(df)}")
print(f"Tổng số đặc trưng: {len(df.columns)}")
print(f"\nVài dòng đầu tiên:")
print(df.head())

# Hiển thị kiểu dữ liệu
print(f"\n\nKiểu dữ liệu:")
print(df.dtypes)

# Xác định các cột phân loại và cột số
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"\n\nCác cột phân loại ({len(categorical_cols)}): {categorical_cols}")
print(f"\nCác cột số ({len(numeric_cols)}): {numeric_cols[:10]}... (tổng: {len(numeric_cols)})")

# Hiển thị thông tin biến mục tiêu
target_col = DATA_CONFIG['target_column']
print(f"\n\nBIẾN MỤC TIÊU: {target_col}")
print(f"Số lượng theo giá trị:\n{df[target_col].value_counts()}")
print(f"Tỷ lệ hủy đặt phòng: {(df[target_col].sum() / len(df)) * 100:.2f}%")

✓ Tải dữ liệu thành công!
  Kích thước: (119390, 32) (dòng, cột)
  Đường dẫn: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\data\hotel_bookings.csv

TỔNG QUAN VỀ TẬP DỮ LIỆU
Tổng số mẫu: 119390
Tổng số đặc trưng: 32

Vài dòng đầu tiên:
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                       

C:\Users\Thanh Dung\AppData\Local\Temp\ipykernel_20448\1205007946.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [5]:
# Hiển thị thông tin các cột phân loại
print("="*80)
print("CÁC CỘT PHÂN LOẠI - THỐNG KÊ CƠ BẢN")
print("="*80)

cat_info = pd.DataFrame({
    'Cột': categorical_cols,
    'Số_giá_trị_khác_nhau': [df[col].nunique() for col in categorical_cols],
    'Số_lượng_thiếu': [df[col].isna().sum() for col in categorical_cols],
    'Tỷ_lệ_thiếu_%': [(df[col].isna().sum() / len(df)) * 100 for col in categorical_cols],
    'Giá_trị_phổ_biến_nhất': [df[col].value_counts().index[0] if len(df[col].value_counts()) > 0 else None for col in categorical_cols],
    'Số_lần_xuất_hiện_nhiều_nhất': [df[col].value_counts().iloc[0] if len(df[col].value_counts()) > 0 else 0 for col in categorical_cols]
})

cat_info = cat_info.sort_values('Số_giá_trị_khác_nhau', ascending=False)
print(cat_info.to_string(index=False))
print(f"\nTổng số đặc trưng phân loại: {len(cat_info)}")

CÁC CỘT PHÂN LOẠI - THỐNG KÊ CƠ BẢN
                    Cột  Số_giá_trị_khác_nhau  Số_lượng_thiếu  Tỷ_lệ_thiếu_% Giá_trị_phổ_biến_nhất  Số_lần_xuất_hiện_nhiều_nhất
reservation_status_date                   926               0       0.000000            2015-10-21                         1461
                country                   177             488       0.408744                   PRT                        48590
     assigned_room_type                    12               0       0.000000                     A                        74053
     arrival_date_month                    12               0       0.000000                August                        13877
     reserved_room_type                    10               0       0.000000                     A                        85994
         market_segment                     8               0       0.000000             Online TA                        56477
   distribution_channel                     5               0       

## Phần 3: Kết Quả Phân Tích Phân Phối

Tóm tắt phân phối của từng biến phân loại, bao gồm số giá trị khác nhau và tỷ lệ dữ liệu thiếu.


In [6]:
# Loại bỏ cột target khỏi danh sách phân tích nếu có
analysis_categorical_cols = [col for col in categorical_cols if col != target_col]

# Chạy toàn bộ pipeline EDA biến phân loại
print("Bắt đầu pipeline EDA biến phân loại...\n")

eda_results = run_categorical_eda(
    df,
    target_column=target_col,
    categorical_columns=analysis_categorical_cols,
    output_dir=EDA_CONFIG['output_dir']
)

print("\n✓ Pipeline EDA biến phân loại đã hoàn thành!")
print(f"✓ Tất cả biểu đồ và báo cáo đã được lưu vào: {EDA_CONFIG['output_dir']}")

Bắt đầu pipeline EDA biến phân loại...

[INFO] Starting Categorical EDA Pipeline...
[STEP 1] Analyzing categorical distribution...
[INFO] Categorical distribution report saved to reports/eda/categorical_distribution_analysis.txt
[STEP 2] Analyzing categorical features - target relationship...
[INFO] Categorical-target relationship report saved to reports/eda/categorical_target_analysis.txt
[STEP 3] Generating visualizations...


c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\modules\eda.py:463: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=value_counts.index, y=value_counts.values, ax=ax, palette='viridis')


[INFO] Categorical distribution plot saved to reports/eda/cat_dist_hotel.png


c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\modules\eda.py:463: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=value_counts.index, y=value_counts.values, ax=ax, palette='viridis')
c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\modules\eda.py:471: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


[INFO] Categorical distribution plot saved to reports/eda/cat_dist_arrival_date_month.png


...

[INFO] Categorical distribution plot saved to reports/eda/cat_dist_customer_type.png


c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\modules\eda.py:463: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=value_counts.index, y=value_counts.values, ax=ax, palette='viridis')


[INFO] Categorical distribution plot saved to reports/eda/cat_dist_reservation_status.png


c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\modules\eda.py:463: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=value_counts.index, y=value_counts.values, ax=ax, palette='viridis')
c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\modules\eda.py:471: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


[INFO] Categorical distribution plot saved to reports/eda/cat_dist_reservation_status_date.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_hotel.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_arrival_date_month.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_meal.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_country.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_market_segment.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_distribution_channel.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_reserved_room_type.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_assigned_room_type.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_deposit_type.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_customer_type.png
[INFO] Categorical-target plot saved to reports/eda/cat_target_reservation_status.png
[INFO] Cat

## Phần 4: Phân Tích Mối Quan Hệ Với Biến Mục Tiêu

Phân tích thống kê về cách mỗi biến phân loại tương quan với biến mục tiêu (is_canceled).
Bao gồm kiểm định Chi-Square và đo lường Cramér's V để xác định các liên kết mạnh nhất.


In [7]:
# Hiển thị kết quả phân tích phân phối
dist_results = eda_results['distribution_analysis']

print("="*100)
print("CÁC BIẾN PHÂN LOẠI - PHÂN TÍCH PHÂN PHỐI")
print("="*100)

for col, stats in dist_results.items():
    print(f"\n{'─'*100}")
    print(f"📊 CỘT: {col.upper()}")
    print(f"{'─'*100}")
    print(f"  Số giá trị khác nhau: {stats['unique_values']}")
    print(f"  Dữ liệu thiếu: {stats['missing_count']} ({stats['missing_percentage']:.2f}%)")
    print(f"  Giá trị phổ biến nhất: {stats['most_common']}")
    print(f"  Giá trị ít gặp nhất: {stats['least_common']}")
    print(f"\n  Phân phối giá trị:")

    for value, count in stats['value_counts'].items():
        pct = (count / len(df)) * 100
        bar_length = int(pct / 2)
        bar = '█' * bar_length
        print(f"    {str(value)[:30]:30s} {count:8d} ({pct:6.2f}%) {bar}")

CÁC BIẾN PHÂN LOẠI - PHÂN TÍCH PHÂN PHỐI

────────────────────────────────────────────────────────────────────────────────────────────────────
📊 CỘT: HOTEL
────────────────────────────────────────────────────────────────────────────────────────────────────
  Số giá trị khác nhau: 2
  Dữ liệu thiếu: 0 (0.00%)
  Giá trị phổ biến nhất: City Hotel
  Giá trị ít gặp nhất: Resort Hotel

  Phân phối giá trị:
    City Hotel                        79330 ( 66.45%) █████████████████████████████████
    Resort Hotel                      40060 ( 33.55%) ████████████████

────────────────────────────────────────────────────────────────────────────────────────────────────
📊 CỘT: ARRIVAL_DATE_MONTH
────────────────────────────────────────────────────────────────────────────────────────────────────
  Số giá trị khác nhau: 12
  Dữ liệu thiếu: 0 (0.00%)
  Giá trị phổ biến nhất: August
  Giá trị ít gặp nhất: January

  Phân phối giá trị:
    August                            13877 ( 11.62%) █████
    July 

## Phần 5: Phát Hiện Quan Trọng & Tóm Tắt

Bảng xếp hạng cuối cùng của các biến phân loại theo tầm quan trọng dự đoán cho pipeline học máy.

In [8]:
# Hiển thị kết quả phân tích mối quan hệ với biến mục tiêu
target_results = eda_results['target_relationship']

print("="*120)
print("CÁC BIẾN PHÂN LOẠI - PHÂN TÍCH MỐI QUAN HỆ VỚI MỤC TIÊU (is_canceled)")
print("="*120)

# Tạo bảng tóm tắt
summary_data = []
for col, stats in target_results.items():
    summary_data.append({
        'Đặc_trưng': col,
        'Chi-Square': f"{stats['chi2_statistic']:.4f}",
        'P-Value': f"{stats['p_value']:.6f}",
        "Cramér's V": f"{stats['cramers_v']:.4f}",
        'Mức_ý_nghĩa': '✓✓✓ Mạnh' if stats['p_value'] < 0.001 else ('✓✓ Trung bình' if stats['p_value'] < 0.05 else '✗ Yếu')
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Cramér's V", ascending=False, key=lambda x: pd.to_numeric(x, errors='coerce'))

print("\nBảng tóm tắt kiểm định thống kê (sắp xếp theo Cramér's V - độ mạnh liên hệ):")
print(summary_df.to_string(index=False))

print("\n\nDiễn giải:")
print("  • Cramér's V: 0 = không liên hệ, 1 = liên hệ hoàn toàn")
print("  • P-Value < 0.05: Có ý nghĩa thống kê")
print("  • P-Value < 0.001: Ý nghĩa thống kê rất mạnh")

# Kết quả chi tiết từng đặc trưng
print("\n\n" + "="*120)
print("KẾT QUẢ CHI TIẾT TỪNG ĐẶC TRƯNG")
print("="*120)

for col, stats in target_results.items():
    print(f"\n{'─'*120}")
    print(f"📈 {col.upper()}")
    print(f"{'─'*120}")
    print(f"  Thống kê Chi-Square: {stats['chi2_statistic']:.4f}")
    print(f"  P-Value: {stats['p_value']:.6f}")
    print(f"  Cramér's V: {stats['cramers_v']:.4f}")

    if stats['p_value'] < 0.001:
        print(f"  → Ý nghĩa thống kê RẤT MẠNH (p < 0.001) ✓✓✓")
    elif stats['p_value'] < 0.05:
        print(f"  → Ý nghĩa thống kê TRUNG BÌNH (p < 0.05) ✓✓")
    else:
        print(f"  → Ý nghĩa thống kê YẾU (p ≥ 0.05) ✗")

    print(f"\n  Tỷ lệ hủy theo danh mục:")

    # Trích xuất dữ liệu tỷ lệ hủy một cách an toàn
    try:
        cancel_rates_dict = stats['cancellation_by_category']
        if isinstance(cancel_rates_dict, dict):
            total_bookings = cancel_rates_dict.get('total_bookings', {})
            cancellations = cancel_rates_dict.get('cancellations', {})
            rates = cancel_rates_dict.get('cancellation_rate', {})

            if isinstance(total_bookings, dict):
                for cat_val in total_bookings.keys():
                    total = total_bookings.get(cat_val, 0)
                    cancel = cancellations.get(cat_val, 0)
                    rate = rates.get(cat_val, 0)
                    print(f"      {str(cat_val)[:25]:25s}: {int(total):8d} đặt phòng, {int(cancel):8d} hủy ({rate*100:6.2f}%)")
    except Exception as e:
        print(f"      (Không thể phân tích dữ liệu tỷ lệ hủy: {e})")

CÁC BIẾN PHÂN LOẠI - PHÂN TÍCH MỐI QUAN HỆ VỚI MỤC TIÊU (is_canceled)

Bảng tóm tắt kiểm định thống kê (sắp xếp theo Cramér's V - độ mạnh liên hệ):
              Đặc_trưng  Chi-Square  P-Value Cramér's V Mức_ý_nghĩa
     reservation_status 119390.0000 0.000000     1.0000    ✓✓✓ Mạnh
reservation_status_date  28423.0595 0.000000     0.4879    ✓✓✓ Mạnh
           deposit_type  27677.3292 0.000000     0.4815    ✓✓✓ Mạnh
                country  15434.6815 0.000000     0.3603    ✓✓✓ Mạnh
         market_segment   8497.2241 0.000000     0.2668    ✓✓✓ Mạnh
     assigned_room_type   4918.6934 0.000000     0.2030    ✓✓✓ Mạnh
   distribution_channel   3745.7941 0.000000     0.1771    ✓✓✓ Mạnh
                  hotel   2224.9249 0.000000     0.1365    ✓✓✓ Mạnh
          customer_type   2222.5042 0.000000     0.1364    ✓✓✓ Mạnh
     reserved_room_type    647.8351 0.000000     0.0737    ✓✓✓ Mạnh
     arrival_date_month    588.6918 0.000000     0.0702    ✓✓✓ Mạnh
                   meal    304.2362 

In [9]:
# Tạo và lưu báo cáo phát hiện quan trọng
print("\n\n" + "="*100)
print("PHÁT HIỆN QUAN TRỌNG & KHUYẾN NGHỊ CHO BƯỚC TIỀN XỬ LÝ")
print("="*100)

# Tạo bảng xếp hạng tầm quan trọng
importance_data = []
for col, stats in target_results.items():
    col_stats = dist_results.get(col, {})
    importance_data.append({
        'Feature': col,
        'CramersV': stats['cramers_v'],
        'PValue': stats['p_value'],
        'Unique': col_stats.get('unique_values', 0),
        'Missing_Pct': col_stats.get('missing_percentage', 0),
        'Recommendation': 'Ưu tiên cao' if stats['cramers_v'] > 0.15 else ('Ưu tiên trung bình' if stats['cramers_v'] > 0.05 else 'Ưu tiên thấp')
    })

importance_df = pd.DataFrame(importance_data)
importance_df = importance_df.sort_values("CramersV", ascending=False)

print("\n📊 CÁC BIẾN PHÂN LOẠI XẾP HẠNG THEO TẦM QUAN TRỌNG:")
print(importance_df.to_string(index=False))

# Xuất báo cáo tóm tắt dạng markdown
findings_md = """# Phân Tích EDA Biến Phân Loại - Phát Hiện Quan Trọng

## Tóm Tắt
Báo cáo này tóm tắt kết quả phân tích khám phá dữ liệu (EDA) các biến phân loại trong tập dữ liệu đặt phòng khách sạn.
Phân tích tập trung vào việc xác định biến phân loại nào có mối quan hệ mạnh nhất với việc hủy đặt phòng.

## Tổng Quan Tập Dữ Liệu
- **Tổng số bản ghi**: {total_records}
- **Tỷ lệ hủy đặt phòng**: {cancel_rate:.2f}%
- **Số biến phân loại được phân tích**: {num_features}

## Xếp Hạng Tầm Quan Trọng Của Đặc Trưng (theo Cramér's V)
| Hạng | Đặc trưng | Cramér's V | P-Value | Số giá trị khác nhau | Tỷ lệ thiếu % | Ưu tiên |
|------|-----------|-----------|---------|----------------------|----------------|---------|
{table_rows}

## Hướng Dẫn Diễn Giải
- **Cramér's V**: Đo độ mạnh liên hệ giữa biến phân loại và mục tiêu (0=không liên hệ, 1=hoàn toàn liên hệ)
- **P-Value**: Mức ý nghĩa thống kê (< 0.05 = có ý nghĩa)
- **Ưu tiên cao (V > 0.15)**: Khả năng dự đoán mạnh, PHẢI mã hóa cẩn thận
- **Ưu tiên trung bình (0.05 < V ≤ 0.15)**: Khả năng dự đoán trung bình, nên đưa vào mô hình
- **Ưu tiên thấp (V ≤ 0.05)**: Khả năng dự đoán yếu, có thể bỏ qua

## Các Đặc Trưng Hàng Đầu Cần Mã Hóa (Ưu Tiên Cao)
{high_priority_features}

## Khuyến Nghị Cho Giai đoạn tiếp theo (Tiền Xử Lý)
1. **Chiến lược**:
   - Dùng One-Hot Encoding cho biến ưu tiên cao có ít giá trị khác nhau
   - Dùng Label Encoding cho biến có nhiều giá trị duy nhất (ví dụ: country)
   - Cân nhắc embedding cho biến có quá nhiều giá trị duy nhất

2. **Lưu ý về chuẩn hóa (Scaling)**:
   - Đặc trưng sau khi One-Hot Encoding đã nằm trong khoảng [0, 1]
   - Sau khi mã hóa, chỉ áp dụng scaling cho các cột số

## Các File Đầu Ra Đã Tạo
Tất cả biểu đồ và báo cáo chi tiết đã được lưu vào: `reports/eda/`
- `categorical_distribution_analysis.txt`: Thống kê phân phối chi tiết
- `categorical_target_analysis.txt`: Kết quả kiểm định thống kê
- `categorical_association_heatmap.png`: Xếp hạng trực quan tầm quan trọng đặc trưng
- `cat_dist_*.png`: Biểu đồ phân phối từng biến phân loại
- `cat_target_*.png`: Tỷ lệ hủy theo danh mục

## Bước Tiếp Theo
Chuyển báo cáo này cho Thành viên 1 để tích hợp logic mã hóa vào `modules/preprocessing.py`
dựa trên xếp hạng tầm quan trọng và các khuyến nghị ở trên.
"""

# Chuẩn bị các dòng của bảng
table_rows = ""
for idx, (_, row) in enumerate(importance_df.iterrows(), 1):
    cr_v = row['CramersV']
    table_rows += f"| {idx} | {row['Feature']} | {cr_v:.4f} | {row['PValue']:.6f} | {row['Unique']} | {row['Missing_Pct']:.2f}% | {row['Recommendation']} |\n"

# Chuẩn bị danh sách đặc trưng ưu tiên cao
high_priority = importance_df[importance_df['Recommendation'] == 'Ưu tiên cao']
high_priority_features = "\n".join([f"- **{row['Feature']}** (Cramér's V = {row['CramersV']:.4f})" for _, row in high_priority.iterrows()])
if not high_priority_features:
    high_priority_features = "- Không tìm thấy đặc trưng ưu tiên cao (tất cả Cramér's V <= 0.15)"

# Chuẩn bị khuyến nghị xử lý dữ liệu thiếu
missing_data_recommendations = ""
for col in analysis_categorical_cols:
    col_info = dist_results.get(col, {})
    missing_pct = col_info.get('missing_percentage', 0)
    if missing_pct > 0:
        missing_data_recommendations += f"- **{col}**: thiếu {missing_pct:.2f}% - Dùng Mode Imputation (giá trị xuất hiện nhiều nhất)\n"

if not missing_data_recommendations:
    missing_data_recommendations = "- Không phát hiện dữ liệu thiếu đáng kể trong các biến phân loại\n"

# Điền vào template
findings_md = findings_md.format(
    total_records=len(df),
    cancel_rate=(df[target_col].sum() / len(df)) * 100,
    num_features=len(analysis_categorical_cols),
    table_rows=table_rows,
    high_priority_features=high_priority_features,
    missing_data_recommendations=missing_data_recommendations
)

# Lưu ra file
output_path = os.path.join(EDA_CONFIG['output_dir'], 'CATEGORICAL_EDA_FINDINGS.md')
os.makedirs(EDA_CONFIG['output_dir'], exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(findings_md)

print(f"\n✓ Báo cáo phát hiện đã được lưu vào: {output_path}")

# Hiển thị tóm tắt
print("\n" + "="*100)
print("✓ PHÂN TÍCH EDA BIẾN PHÂN LOẠI HOÀN TẤT!")
print("="*100)
print(f"\nCác file đầu ra được tạo trong {EDA_CONFIG['output_dir']}:")
print("  ✓ categorical_distribution_analysis.txt")
print("  ✓ categorical_target_analysis.txt")
print("  ✓ categorical_association_heatmap.png")
print("  ✓ Biểu đồ phân phối từng cột (cat_dist_*.png)")
print("  ✓ Biểu đồ tỷ lệ hủy theo danh mục (cat_target_*.png)")
print("  ✓ CATEGORICAL_EDA_FINDINGS.md (Tóm tắt cho nhóm tiền xử lý)")



PHÁT HIỆN QUAN TRỌNG & KHUYẾN NGHỊ CHO BƯỚC TIỀN XỬ LÝ

📊 CÁC BIẾN PHÂN LOẠI XẾP HẠNG THEO TẦM QUAN TRỌNG:
                Feature  CramersV        PValue  Unique  Missing_Pct     Recommendation
     reservation_status  1.000000  0.000000e+00       3     0.000000        Ưu tiên cao
reservation_status_date  0.487923  0.000000e+00     926     0.000000        Ưu tiên cao
           deposit_type  0.481480  0.000000e+00       3     0.000000        Ưu tiên cao
                country  0.360292  0.000000e+00     177     0.408744        Ưu tiên cao
         market_segment  0.266781  0.000000e+00       8     0.000000        Ưu tiên cao
     assigned_room_type  0.202974  0.000000e+00      12     0.000000        Ưu tiên cao
   distribution_channel  0.177128  0.000000e+00       5     0.000000        Ưu tiên cao
                  hotel  0.136513  0.000000e+00       2     0.000000 Ưu tiên trung bình
          customer_type  0.136439  0.000000e+00       4     0.000000 Ưu tiên trung bình
     reserv